In [2]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Constants
UNANSWERABLE_TOKEN = "[BRAK_ODPOWIEDZI]"
MODEL_KIND = "base"

# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained(f'allegro/plt5-{MODEL_KIND}', legacy=False)
tokenizer.add_tokens([UNANSWERABLE_TOKEN])
model = T5ForConditionalGeneration.from_pretrained(f'allegro/plt5-{MODEL_KIND}')
model.resize_token_embeddings(len(tokenizer))

# Move model to CUDA
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# Test inputs
inputs = tokenizer("Example input", return_tensors="pt").to(device)
labels = tokenizer("Example output", return_tensors="pt").input_ids.to(device)

# Forward pass
outputs = model(input_ids=inputs.input_ids, labels=labels)
print(outputs)

# Check for NaNs in embeddings and inputs
embeddings = model.get_input_embeddings().weight.data
print("Embeddings after resize (should not contain NaNs):", torch.isnan(embeddings).any())
print("Inputs (should not contain NaNs):", torch.isnan(inputs.input_ids).any())
print("Labels (should not contain NaNs):", torch.isnan(labels).any())

print(outputs.loss)

Seq2SeqLMOutput(loss=tensor(16.1770, device='cuda:0', grad_fn=<NllLossBackward0>), logits=tensor([[[-149.7673,  -15.2614,  -27.3672,  ...,  -30.3327,    7.1881,
           -83.4996],
         [ -87.0950,  -11.1824,   -8.3990,  ...,   -8.1408,  -34.2834,
           -43.4390],
         [-113.1793,   -4.3365,  -15.9299,  ...,  -19.5681,  -44.9293,
           -58.1549],
         ...,
         [-195.1959,  -11.1191,  -43.4416,  ...,  -43.1148,  -75.0038,
          -109.6791],
         [-192.4121,   -8.7038,  -44.7336,  ...,  -43.6849,  -72.8442,
          -109.8279],
         [-197.9768,  -10.3834,  -45.8841,  ...,  -44.0242,  -76.6241,
          -112.4943]]], device='cuda:0', grad_fn=<UnsafeViewBackward0>), past_key_values=((tensor([[[[-0.1138,  0.7574,  0.6319,  ...,  1.1968, -0.3768,  0.3205],
          [-0.3922,  0.2604,  0.3558,  ...,  0.4664, -0.5402, -0.4563],
          [ 0.3703,  1.1511, -1.3338,  ...,  0.4119, -0.2172,  0.2095],
          ...,
          [ 0.3756,  0.5864,  0.2454, 

In [4]:
import os
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

UNANSWERABLE_TOKEN = "[BRAK_ODPOWIEDZI]"
MODEL_KIND = "small"
MODEL_FOLDER = "../models/plt5-original-small"

def download_and_save_plt5():
    tokenizer = T5Tokenizer.from_pretrained(f'allegro/plt5-{MODEL_KIND}', legacy=False)
    model = T5ForConditionalGeneration.from_pretrained(f'allegro/plt5-{MODEL_KIND}')
    tokenizer.add_tokens([UNANSWERABLE_TOKEN])
    model.resize_token_embeddings(len(tokenizer))

    if not os.path.exists(MODEL_FOLDER):
        os.makedirs(MODEL_FOLDER)

    tokenizer.save_pretrained(MODEL_FOLDER)
    model.save_pretrained(MODEL_FOLDER)

def load_plt5(model_folder):
    tokenizer = T5Tokenizer.from_pretrained(model_folder)
    model = T5ForConditionalGeneration.from_pretrained(model_folder)
    return tokenizer, model

# Download and save the model
download_and_save_plt5()

# Load the model
tokenizer, model = load_plt5(MODEL_FOLDER)

# Move model to CUDA
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

# Test inputs
inputs = tokenizer("Example input", return_tensors="pt").to(device)
labels = tokenizer("Example output", return_tensors="pt").input_ids.to(device)

# Forward pass
outputs = model(input_ids=inputs.input_ids, labels=labels)
print(outputs)

# Check for NaNs in embeddings and inputs
embeddings = model.get_input_embeddings().weight.data
print("Embeddings after resize (should not contain NaNs):", torch.isnan(embeddings).any())
print("Inputs (should not contain NaNs):", torch.isnan(inputs.input_ids).any())
print("Labels (should not contain NaNs):", torch.isnan(labels).any())

# Additional checks
print("Special tokens:", tokenizer.special_tokens_map)
print("Model config:", model.config)


Seq2SeqLMOutput(loss=tensor(21.3761, device='cuda:0', grad_fn=<NllLossBackward0>), logits=tensor([[[-99.7277, -32.1589, -29.6742,  ..., -32.2528,  -4.7667, -61.8801],
         [-64.3722,   5.1307, -19.0437,  ..., -37.4680, -42.4685, -43.2602],
         [-64.8318,   4.8287, -20.0719,  ..., -35.8161, -41.7448, -42.2242],
         ...,
         [-70.0506,  -6.3271,  -7.5067,  ..., -14.8673, -32.5707, -38.8790],
         [-70.9328,  -2.1656,  -6.9357,  ..., -13.0630, -31.8631, -39.0469],
         [-68.1579, -11.7990,  -3.7408,  ..., -16.6116, -29.8864, -36.5168]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>), past_key_values=((tensor([[[[-0.2199, -0.4390,  0.1116,  ...,  0.1291,  0.6233, -0.2862],
          [-0.7961,  1.0791, -0.4187,  ...,  1.1468,  1.7722, -0.3422],
          [ 0.2313,  0.0656,  1.7133,  ..., -1.6529, -0.5622,  0.0456],
          ...,
          [ 0.6329, -1.3505, -0.2100,  ...,  1.9367,  0.7670,  1.7529],
          [-0.1817, -2.2149, -1.2826,  ..., -2.7933,  0